In [1]:
!pip install pdfplumber pandas openpyxl

import pdfplumber
import pandas as pd
import glob
import re

# --- 1. Harga jual & HPP mapping ---
menu_info = {
    "Arure Matcha Latte": {"harga": 23000, "hpp": 14000},
    "Matcha Cold Whisk": {"harga": 25000, "hpp": 14100},
    "Kopi Arure": {"harga": 18000, "hpp": 10000},
    "Add Oatside Milk": {"harga": 3000, "hpp": 2200},
    "Add Vanilla Syrup": {"harga": 3000, "hpp": 1500},
}

data = []

# --- 2. Baca semua file bon PDF ---
for file in glob.glob("/content/bon/*.pdf"):
    with pdfplumber.open(file) as pdf:
        text = ""
        for page in pdf.pages:
            text += page.extract_text() + "\n"

        # ambil semua item + qty + harga
        items = re.findall(r"(.*?)\s+(\d+)\s+Rp\s([\d\.]+)", text)
        total = re.search(r"TOTAL\s+Rp\s([\d\.]+)", text)

        entry = {"File": file.split("/")[-1]}
        for item, qty, harga in items:
            qty = int(qty)
            item = item.strip()
            if item in menu_info:
                entry[item] = qty
        entry["Total (Rp)"] = int(total.group(1).replace(".", "")) if total else 0
        data.append(entry)

# --- 3. Buat DataFrame detail per struk ---
df = pd.DataFrame(data).fillna(0)

# --- 4. Pivot rekap penjualan ---
rekap_list = []
for col in df.columns:
    if col in menu_info:
        total_qty = df[col].sum()
        harga_jual = menu_info[col]["harga"]
        hpp = menu_info[col]["hpp"]

        omzet = total_qty * harga_jual
        biaya = total_qty * hpp
        profit = omzet - biaya

        rekap_list.append({
            "Menu": col,
            "Qty Terjual": total_qty,
            "Harga Jual": harga_jual,
            "HPP": hpp,
            "Total Omzet": omzet,
            "Total HPP": biaya,
            "Profit Bersih": profit
        })

rekap_df = pd.DataFrame(rekap_list)

# --- 5. Export ke Excel ---
with pd.ExcelWriter("rekap_bon_with_profit.xlsx") as writer:
    df.to_excel(writer, sheet_name="Detail Struk", index=False)
    rekap_df.to_excel(writer, sheet_name="Pivot Penjualan", index=False)

print("✅ Selesai! File 'rekap_bon_with_profit.xlsx' sudah dibuat.")


✅ Selesai! File 'rekap_bon_with_profit.xlsx' sudah dibuat.
